# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined by a Croissant schema using the `mlcroissant` Python library. All dataset entities (record sets, fields, columns) are referenced by their `@id` as per best practice.

### Dataset Source
FAIR² dataset for ordered logistic regression outputs in rangeland management (Kenya).<br>
Croissant schema URL: 
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Metadata as an object
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity uses its `@id` for reference.

**Tip:** Record sets and their fields are referenced by `@id` only.

In [ ]:
# Display available record sets
print("Available record sets (@id):")
for record_set in dataset.record_sets:
    print(f"- {record_set.id} (name: {getattr(record_set, 'name', '-')})")

# Display fields for each record set (by @id)
for record_set in dataset.record_sets:
    print(f"\nFields for record set @id: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        for field in record_set.fields:
            print(f"  - Field @id: {field.id}, name: {getattr(field, 'name', '-')}")
    else:
        print("  (No fields listed)")

## 3. Data Extraction
Load records from record sets as DataFrames using their `@id` fields.

- List the record set `@id`s for extraction.
- Each DataFrame is referenced by the corresponding record set `@id`.

In [ ]:
# Gather record set @id's for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Using the @id for reference
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

print("Extracted DataFrame keys (record set @id):", list(dataframes.keys()))

# Preview columns of each loaded DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nRecord Set @id: {record_set_id} — Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate processing of a numeric field using the record set `@id` and field `@id`. Examples include filtering, normalization, and grouping.

*If there are no numerical fields or if the dataset is empty, this section will illustrate the general approach*.

In [ ]:
# Choose a record set with data and a numeric column
# (Adjust these IDs based on the overview above. If none found, use a placeholder)
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

# Find a record set with numeric field
for record_set_id, df in dataframes.items():
    if not df.empty:
        # Find numeric columns
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                selected_record_set_id = record_set_id
                numeric_field_id = col
                # Find a possible group-by field
                for gc in df.columns:
                    if df[gc].nunique() > 1 and gc != col:
                        group_field_id = gc
                        break
                break
    if selected_record_set_id and numeric_field_id:
        break

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]

    # Drop NaNs for the numeric field
    filtered_df = df[df[numeric_field_id].notnull()]

    if not filtered_df.empty:
        threshold = filtered_df[numeric_field_id].mean()
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records from '{selected_record_set_id}' where {numeric_field_id} > mean (threshold={threshold:.2f}):")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - \
                                                        filtered_df[numeric_field_id].mean()) / \
                                                      filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field if available
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No records pass the numeric filter.")
else:
    print("No numerical fields found for EDA. Please examine the record set overview above for available options.")

## 5. Visualization
Visualize numeric distributions or relationships between fields, referencing all data elements by their `@id`.

- For demonstration, if a DataFrame with a numeric field is available, use a histogram and, if grouping field exists, a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (selected_record_set_id is not None) and (numeric_field_id is not None):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group field exists, plot boxplot
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} grouped by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No suitable numeric fields or grouping variables available for visualization.")

## 6. Conclusion

* We've demonstrated how to explore and process a Croissant-based dataset with `mlcroissant`, referencing all entities by their `@id` for transparency and reproducibility.
* For this dataset, record sets, fields, and their values can be inspected and manipulated dynamically using Python and Croissant standards.
* Further analysis can leverage the extracted DataFrames, always referencing and documenting data elements by their `@id`.

_For questions or issues, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/)._
